# 04 실습: State · RunnableConfig · 체크포인터
-  LangGraph 1.2.x

## 왜 이걸 배워야 할까? — 사내 HR 챗봇을 운영하다 생긴 일과 해결

- 계속 남아야 하는 건 State에, 실행할 때마다 바뀌는 건 config에 넣는다.

| # | 신고 내용 | 원인 | 해결 |
|---|---|---|---|
| 1 | "방금 이름 말했는데 또 물어봐요." | 대화를 저장할 곳이 없음 | 체크포인터를 붙인다 |
| 1 | "물어본 적 없는 연봉 얘기가 나와요." | 모든 직원이 같은 thread_id | 사용자·대화방별 thread_id |
| 1 | "배포 후 어제 대화가 전부 사라졌어요." | 메모리에만 저장(InMemorySaver) | 파일·DB에 저장(SqliteSaver) |
| 2 | "'공손하게'를 한 번 켰더니 계속 공손해요." | 이번만 쓸 설정을 State에 넣음 | 설정은 config로 |
| 3 | 보안팀: "저장 DB에 API 키가 평문으로 있어요." | 키를 State에 넣음 | 키·DB 연결은 context로 |
| 4 | "검색어를 고쳤는데 다시 돌려도 답이 같아요." | 수정 **전** 지점에서 재실행 | `update_state`가 돌려준 config로 재실행 |
| 5 | "승인할 때 본 주문번호와 실제 번호가 달라요." | 재개 시 노드가 처음부터 다시 실행 | `@task`로 감싸거나 노드를 나눈다 |

### 해결 코드 한눈에

**1. 기억하게 하기: 체크포인터**
```python
app = builder.compile(checkpointer=InMemorySaver())
```

**2. 사람·대화방마다 따로 기억하기: thread_id 설계**
```python
cfg = {"configurable": {"thread_id": f"{user_id}:{session_id}"}}   # 매번 uuid ✗, 모두 "shared" ✗
app.invoke(입력, cfg)
```

**3. 이번만 쓸 설정은 config로**
```python
app.invoke(입력, {"configurable": {"thread_id": tid, "tone": "공손"}})   # 다음 호출엔 남지 않음
# 노드에서: config["configurable"].get("tone", "기본")
```

**4. 재시작해도 남기기: SqliteSaver**
```python
with SqliteSaver.from_conn_string("checkpoints.db") as saver:   # 반드시 with
    app = builder.compile(checkpointer=saver)
```

**5. 비밀 값·연결 객체는 context로**
```python
@dataclass
class Deps:
    db: object
    api_key: str

g = StateGraph(MyState, context_schema=Deps)
app.invoke(입력, cfg, context=Deps(db=conn, api_key=os.environ["API_KEY"]))
# 노드에서: runtime.context.api_key   → 체크포인터에 저장되지 않음
```

**6. 과거 지점에서 고쳐 다시 실행: 돌려받은 config 사용**
```python
new_cfg = app.update_state(target.config, {"query": "연차 이월"}, as_node="make_query")
app.invoke(None, new_cfg)          # target.config ✗
```

**7. 재개해도 값이 바뀌지 않게: @task**
```python
@task
def new_order_no():
    return f"ORD-{random.randint(1000, 9999)}"

def create(state):
    no = new_order_no().result()   # 재개 때는 저장된 값을 돌려줌
    ok = interrupt(f"{no} 결제할까요?")
    return {"order_no": no, "ok": ok}
```

> 일곱 가지 모두 **"무엇을(State/config/context), 어디에(메모리/파일), 누구 단위로(thread_id) 기억하느냐"**의 설계 문제입니다.
> 아래 섹션에서 각각을 직접 재현하고 고쳐 봅니다.

In [ ]:
# !pip install -q langgraph langgraph-checkpoint-sqlite
from typing import Annotated, TypedDict
from operator import add
from dataclasses import dataclass
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime

# [Part1] State 기억하기 
- 그래프를 `compile(checkpointer=InMemorySaver())`로 만들고, 실행할 때 `thread_id`를 주면 사용자별로 State(대화)를 기억할 수 있다.
    - 서브그래프에서는 compile(checkpointer=True)처럼 값 대신 True를 쓸 수 있습니다. 
    - "부모의 체크포인터를 쓰면서 따로 기억하라"는 뜻이고, 부모 그래프에는 쓸 수 없습니다

- from langgraph.checkpoint.sqlite import SqliteSaver
- with SqliteSaver.from_conn_string("chat.db") as saver:      # with 필수
    - app = g.compile(checkpointer=saver)

| 체크포인터 | 저장 위치 | 설치 | 언제 |
|---|---|---|---|
| `InMemorySaver` | 메모리 (재시작하면 사라짐) | 기본 포함 | 개발, 테스트, 수업 |
| `SqliteSaver` / `AsyncSqliteSaver` | SQLite 파일 | `pip install langgraph-checkpoint-sqlite` | 로컬, 서버 한 대 |
| `PostgresSaver` / `AsyncPostgresSaver` | PostgreSQL | `pip install langgraph-checkpoint-postgres` | 운영, 서버 여러 대 |
| Redis, MongoDB 등 | 각 DB | `langgraph-checkpoint-redis`, `langgraph-checkpoint-mongodb` 등 별도 패키지 | 이미 그 DB를 쓰는 회사 |

In [ ]:
# MessagesState: 미리 만들어 둔 대화 메시지를 쌓는 State
# class State(TypedDict):
#    messages: Annotated[list, add_messages]
# https://wikidocs.net/293528


from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver

def bot(state):
    지난질문 = [m.content for m in state["messages"][:-1] if m.type == "human"]
    return {"messages": [("ai", f"이전 질문: {지난질문}")]}

g = StateGraph(MessagesState)
g.add_node("bot", bot)
g.add_edge(START, "bot")

memory = InMemorySaver()                    # ← 저장소를 변수로 받아 둔다
app = g.compile(checkpointer=memory)

def ask(thread_id, 질문):
    cfg = {"configurable": {"thread_id": thread_id}}
    답 = app.invoke({"messages": [("user", 질문)]}, cfg)["messages"][-1].content
    print(f"[{thread_id}] {질문} → {답}")

def show(thread_id):                        # ← 저장된 대화 보기
    cfg = {"configurable": {"thread_id": thread_id}}
    for m in app.get_state(cfg).values["messages"]:
        print(f"  {m.type:>5}: {m.content}")

ask("kim", "내 이름은 철수야")
ask("lee", "내 이름은 영희야")
ask("kim", "내 이름이 뭐야?")
ask("lee", "내 이름이 뭐야?")
ask("lee", "나 영희는 여름을 좋아해")

[kim] 내 이름은 철수야 → 이전 질문: []
[lee] 내 이름은 영희야 → 이전 질문: []
[kim] 내 이름이 뭐야? → 이전 질문: ['내 이름은 철수야']
[lee] 내 이름이 뭐야? → 이전 질문: ['내 이름은 영희야']
[lee] 나 영희는 여름을 좋아해 → 이전 질문: ['내 이름은 영희야', '내 이름이 뭐야?']


In [24]:
# 저장된 스레드 목록
{c.config["configurable"]["thread_id"] for c in memory.list(None)}   # {'kim', 'lee'}

# 스레드별 대화
show("lee")

  human: 내 이름은 영희야
     ai: 이전 질문: []
  human: 내 이름이 뭐야?
     ai: 이전 질문: ['내 이름은 영희야']
  human: 나 영희는 여름을 좋아해
     ai: 이전 질문: ['내 이름은 영희야', '내 이름이 뭐야?']


In [26]:
cfg = {"configurable": {"thread_id": "lee"}}
app.get_state(cfg).values

{'messages': [HumanMessage(content='내 이름은 영희야', additional_kwargs={}, response_metadata={}, id='d5943cde-3321-4b1c-83f2-acaa5e52e4b1'),
  AIMessage(content='이전 질문: []', additional_kwargs={}, response_metadata={}, id='3126cd15-e84b-45ad-aa8c-826f61de0a42', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='3611f22a-4b81-41ff-8f5c-425c1a767c30'),
  AIMessage(content="이전 질문: ['내 이름은 영희야']", additional_kwargs={}, response_metadata={}, id='58db6120-bb96-4f58-84d1-9d996c50a287', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='나 영희는 여름을 좋아해', additional_kwargs={}, response_metadata={}, id='d91d8900-2677-43ff-9612-df9f126c8514'),
  AIMessage(content="이전 질문: ['내 이름은 영희야', '내 이름이 뭐야?']", additional_kwargs={}, response_metadata={}, id='8834f970-fad5-4394-89d2-9d0934bc80bd', tool_calls=[], invalid_tool_calls=[])]}

In [27]:
# 저장된 스레드 목록 전체 보기
{c.config["configurable"]["thread_id"] for c in memory.list(None)}

{'kim', 'lee'}

In [28]:
# 저장소 안 보기
memory.storage.keys()

dict_keys(['kim', 'lee'])

In [36]:
# https://wikidocs.net/321315

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("openai:gpt-4.1-mini")

def chatbot_node(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile(checkpointer=InMemorySaver())

while True:
    아이디 = input("로그인 아이디 (종료: quit): ")
    if 아이디 == "quit":
        break
    config = {"configurable": {"thread_id": 아이디}}     # ← 로그인 아이디 = thread_id

    while True:
        질문 = input(f"{아이디}: ")
        if 질문 == "logout":
            break
        result = graph.invoke({"messages": [("user", 질문)]}, config)
        print("AI:", result["messages"][-1].content)


AI: 비숑(Bichon)은 작고 귀여운 외모를 가진 반려견 종류 중 하나로, 프랑스와 지중해 지역에서 유래했습니다. 비숑 프리제(Bichon Frisé)가 가장 유명하며, 부드럽고 곱슬곱슬한 흰 털이 특징입니다. 성격은 명랑하고 친근하며 사람과 잘 어울려 가족견으로 인기가 많습니다. 또한 지능이 높아 훈련하기 쉽고, 비교적 활동적인 편입니다.
AI: 비숑 프리제의 대표적인 털색은 순백색(흰색)입니다. 간혹 크림빛이나 살짝 아이보리 톤이 섞인 경우도 있지만, 대체로 깨끗한 흰색 털이 가장 일반적이고 선호됩니다.
AI: 안녕하세요! 교통수단에 대해 궁금하신가요? 교통수단은 사람이나 물건을 한 장소에서 다른 장소로 이동시키기 위해 사용하는 다양한 방법이나 도구를 의미합니다. 대표적인 교통수단으로는 다음과 같은 것들이 있어요:

1. **자동차**: 가장 일반적인 도로 교통수단으로, 개인이나 소형, 대형 차량이 포함됩니다.
2. **버스**: 많은 사람을 한꺼번에 태울 수 있는 대중교통 수단입니다.
3. **지하철/전철**: 도시 내에서 빠르게 이동할 수 있는 지하철이나 전철입니다.
4. **기차**: 장거리 이동에 사용되는 철도 교통수단입니다.
5. **자전거**: 개인용 친환경 교통수단입니다.
6. **오토바이/모터사이클**: 빠르게 이동할 수 있고 교통체증을 피하기 쉬운 2륜 차량입니다.
7. **택시/카카오택시**: 승객이 원하는 목적지까지 데려다 주는 유료 교통수단입니다.
8. **비행기**: 장거리, 국제 이동에 사용되는 항공 교통수단입니다.
9. **배/페리**: 바다나 강 등 수로를 이용한 교통수단입니다.
10. **도보**: 가장 기본적인 이동 방법으로, 걷기를 의미합니다.

원하시는 교통수단에 대해 더 상세한 정보가 필요하시면 말씀해 주세요!
AI: 가장 저렴한 교통수단은 보통 **도보**입니다. 걷는 데에는 별도의 비용이 들지 않기 때문에 누구나 부담 없이 이용할 수 있죠.

그 다음으로 저렴한 교통수단으로는 다음이 있습니다:

- **자전거**

In [ ]:
! pip install langgraph.checkpoint.sqlite

In [ ]:
import sqlite3
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver           # ← 변경

model = init_chat_model("openai:gpt-4.1-mini")

def chatbot_node(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

conn = sqlite3.connect("chat.db", check_same_thread=False)    # ← 변경: 파일에 저장
graph = builder.compile(checkpointer=SqliteSaver(conn))       # ← 변경

while True:
    아이디 = input("로그인 아이디 (종료: quit): ")
    if 아이디 == "q":
        break
    config = {"configurable": {"thread_id": 아이디}}

    while True:
        질문 = input(f"{아이디}: ")
        if 질문 == "logout":
            break
        result = graph.invoke({"messages": [("user", 질문)]}, config)
        print("AI:", result["messages"][-1].content)

## [Part2] config
= 말투(config)는 이어지지 않는다: 로그인할 때마다 새로 준 값이 적용돼요.

In [ ]:
import sqlite3
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

model = init_chat_model("openai:gpt-4.1-mini")

def chatbot_node(state: MessagesState, config):                  # ← config 받기
    말투 = config["configurable"].get("말투", "존댓말")            # ← 말투 꺼내기
    지시 = ("system", f"항상 {말투}로 대답해.")                    # ← LLM에 줄 지시문
    return {"messages": [model.invoke([지시] + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

conn = sqlite3.connect("chat.db", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))

while True:
    아이디 = input("로그인 아이디 (종료: quit): ")
    if 아이디 == "q":
        break
    말투 = input("말투 (반말/존댓말): ")
    config = {"configurable": {"thread_id": 아이디, "말투": 말투}}  # ← 둘 다 config로

    while True:
        질문 = input(f"{아이디}: ")
        if 질문 == "logout":
            break
        result = graph.invoke({"messages": [("user", 질문)]}, config)
        print("AI:", result["messages"][-1].content)

AI: 제주도는 한국 남쪽에 위치한 아름다운 섬이야. 자연 경관이 멋지고, 한라산, 성산 일출봉 같은 명소가 유명하지. 특히 해변과 오름, 그리고 독특한 제주도 문화도 매력적이야. 여행 가보고 싶어?
AI: 제주도 맛집으로는 흑돼지 구이집이 진짜 유명해. '돈사돈'이나 '제주돈주' 같은 곳이 인기 많아. 해산물 좋아하면 성산 근처의 활어회 맛집들도 꼭 가봐야 해. 그리고 제주 전통 음식인 ‘고기국수’도 한 그릇 해봐! 어디 가고 싶은 지역 말해주면 더 추천해줄게.
AI: 멍멍! 🐶 어떻게 도와줄까?
AI: 멍멍! 좋아요! 공 던져줄까? 아니면 산책할래요? 🐕‍🦺 함께 놀자!
AI: 제주도 여행지 추천해 드릴게요.

1. 한라산 국립공원 – 등산 코스가 다양해서 등산 좋아하시면 꼭 가보세요. 정상에서 보는 경치가 아름답습니다.  
2. 성산 일출봉 – 일출 명소로 유명하며, 해뜨는 장면을 보기에 최적의 장소입니다.  
3. 천지연 폭포 – 제주에서 가장 유명한 폭포 중 하나로, 산책하며 자연을 즐기기 좋아요.  
4. 우도 – 제주도 동쪽에 있는 작은 섬으로, 자전거 타면서 경치를 즐길 수 있습니다.  
5. 협재 해수욕장 – 맑은 바닷물과 넓은 백사장이 인상적인 해변입니다.  

이 외에도 원하시는 테마가 있으면 말씀해 주세요!
AI: 제주도의 면적은 약 1,848제곱킬로미터야. 한국에서 가장 큰 섬이고, 제주도 전체를 포함한 면적이야.
AI: 제주도의 대략적인 중심 위치는  
위도: 33.3617° N  
경도: 126.5292° E  
입니다.
AI: 제주도에서 유명한 것들 말해줄게.

1. 한라산 – 제주도 최고봉이자 한국에서 가장 높은 산이야.  
2. 성산 일출봉 – 일출 명소로 엄청 유명하지.  
3. 흑돼지 – 제주도 특산 흑돼지 고기가 진짜 맛있어.  
4. 감귤 – 제주에서 나는 감귤도 인기 많아.  
5. 제주 해녀 – 전통적으로 바다에서 해산물을 채취하는 여성을 말하는데, 제주 문화를 대표해.  
6. 오름 – 작은 화산 언덕들이 많이 분포

## [Part3] API 키를 State에 넣지 않고 context로 넘기는 예제

In [40]:
from typing import TypedDict
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

# 1. context의 모양 정하기
#    State처럼 TypedDict로 "어떤 값을 넘길지" 적어 둔다
class Context(TypedDict):
    api_key: str

# 2. 노드에서 context 받기
#    state 다음에 runtime을 받으면, LangGraph가 context를 runtime 안에 넣어 준다
def bot(state: MessagesState, runtime: Runtime[Context]):
    key = runtime.context["api_key"]          # ← 키는 여기서 꺼내 쓰기만 한다
    return {"messages": [("ai", f"키 끝자리 {key[-4:]}로 호출했어요")]}
    # return에 키를 넣지 않았으므로 State에 키가 들어가지 않는다

# 3. 그래프에 context 모양 알려 주기
g = StateGraph(MessagesState, context_schema=Context)
g.add_node("bot", bot)
g.add_edge(START, "bot")

memory = InMemorySaver()
app = g.compile(checkpointer=memory)

# 4. 실행할 때 context로 키 넘기기
config = {"configurable": {"thread_id": "kim"}}
result = app.invoke(
    {"messages": [("user", "안녕")]},         # 입력   → State (저장됨)
    config,                                    # config → thread_id 등 실행 설정
    context={"api_key": "sk-secret-1234"},     # context → 키 (저장 안 됨)
)
print(result["messages"][-1].content)

# 5. 확인: 저장소에 키가 남았는지
print(app.get_state(config).values)                         # State에는 대화만 있음
print("sk-secret" in str(list(memory.list(None))))         # False → 저장소에 키 없음

키 끝자리 1234로 호출했어요
{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='f4365aad-e759-4877-84c0-742d064dba23'), AIMessage(content='키 끝자리 1234로 호출했어요', additional_kwargs={}, response_metadata={}, id='fdcbb326-8e49-4aa6-974c-4572d9710f2d', tool_calls=[], invalid_tool_calls=[])]}
False


# [Part4] 검색어를 고쳤는데 다시 돌려도 답이 같아요
- update_state
- **저장된 State를 밖에서 직접 고치는 함수**

    app.update_state(config, {"검색어": "부산 날씨"})

- 노드를 실행하지 않고 State 값만 바꾼다
- 고친 내용은 **새 체크포인트**로 저장되고, 그 새 config를 돌려준다
- 이어서 실행할 때는 **돌려받은 config**로 실행한다

    새config = app.update_state(config, {"검색어": "부산 날씨"})
    app.invoke(None, 새config)
    

In [41]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    검색어: str
    답: str

# 진짜 검색 대신 쓰는 가짜 검색 자료
자료 = {"제주 날씨": "제주는 맑음", "부산 날씨": "부산은 비"}

def 검색(state):
    return {"답": 자료.get(state["검색어"], "모름")}

g = StateGraph(State)
g.add_node("검색", 검색)
g.add_edge(START, "검색")
app = g.compile(checkpointer=InMemorySaver())

# 1. 처음 실행: "제주 날씨"로 검색
config = {"configurable": {"thread_id": "1"}}
print("처음:", app.invoke({"검색어": "제주 날씨"}, config)["답"])

# 2. 과거 기록에서 "검색 노드 실행 직전" 지점 찾기
#    s.next = 그 지점에서 다음에 실행될 노드
직전 = next(s for s in app.get_state_history(config) if s.next == ("검색",))

# 3. 그 지점의 검색어를 "부산 날씨"로 고치기
#    update_state는 원래 지점을 바꾸지 않고, 고친 내용을 담은 "새 지점"을 만든다
#    → 그 새 지점의 config를 돌려준다
새config = app.update_state(직전.config, {"검색어": "부산 날씨"})

# 4-1. 잘못된 재실행: 고치기 "전" 지점의 config로 실행
print("잘못:", app.invoke(None, 직전.config)["답"])      # 여전히 제주

# 4-2. 올바른 재실행: update_state가 돌려준 config로 실행
print("올바름:", app.invoke(None, 새config)["답"])        # 부산으로 바뀜

처음: 제주는 맑음
잘못: 제주는 맑음
올바름: 부산은 비


# [Part5] 승인을 interrupt로 받았을 때 본 주문번호와 실제 번호가 달라요"

**원인**: `interrupt` 후 재개하면 노드가 **처음부터 다시** 실행된다. 그래서 번호 발급이 두 번 일어난다.

**해결**: 두 번 하면 안 되는 일(번호 발급, 결제, 메일 발송)은 **앞 노드로 따로 뺀다.**

    def 번호발급(state):                      # 앞 노드: 번호를 State에 저장하고 끝
        return {"주문번호": 새번호()}

    def 승인(state):                          # 뒤 노드: 저장된 번호를 쓰기만 한다
        n = state["주문번호"]
        답 = interrupt(f"주문번호 {n} 승인할까요?")
        return {"결과": f"{n}번 주문 {답}"}

> 노드를 나누기 어려우면 그 함수에 `@task`를 붙여도 같은 효과가 난다.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
# interrupt:  그래프를 멈추고 사람에게 질문을 보여 줌 (노드안)
# Command(resume=...): 사람의 답을 넣고 이어서 실행함 (노드박 invoke할때)


class State(TypedDict):
    주문번호: int
    결과: str

# 부를 때마다 1씩 올라가는 주문번호 발급기 (실제로는 DB에서 번호를 받는 부분)
번호 = [1000]
def 새번호():
    번호[0] += 1
    return 번호[0]


# ===== 문제 버전: 번호 발급과 승인이 한 노드에 있음 =====
def 주문(state):
    n = 새번호()                                  # ① 번호 발급
    답 = interrupt(f"주문번호 {n} 승인할까요?")     # ② 여기서 멈추고 사람에게 보여 줌
    return {"주문번호": n, "결과": f"{n}번 주문 {답}"}
    # 재개하면 노드가 ①부터 "처음부터 다시" 실행된다
    # → 새번호()가 한 번 더 불려 다른 번호가 나온다

g = StateGraph(State)
g.add_node("주문", 주문)
g.add_edge(START, "주문")
app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "a"}}
r = app.invoke({}, cfg)
print("화면:", r["__interrupt__"][0].value)                       # 사람이 본 번호
print("실제:", app.invoke(Command(resume="승인"), cfg)["결과"])    # 실제 처리된 번호


# ===== 해결 버전: 번호 발급과 승인을 노드로 나눔 =====
def 번호발급(state):
    return {"주문번호": 새번호()}                  # 번호를 State에 저장하고 끝
                                                   # → 체크포인트에 기록되므로 다시 실행되지 않음

def 승인(state):
    n = state["주문번호"]                          # 저장된 번호를 꺼내 쓰기만 한다
    답 = interrupt(f"주문번호 {n} 승인할까요?")     # 재개 시 이 노드만 다시 실행
    return {"결과": f"{n}번 주문 {답}"}

g = StateGraph(State)
g.add_node("번호발급", 번호발급)
g.add_node("승인", 승인)
g.add_edge(START, "번호발급")
g.add_edge("번호발급", "승인")
app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "b"}}
r = app.invoke({}, cfg)
print("화면:", r["__interrupt__"][0].value)
print("실제:", app.invoke(Command(resume="승인"), cfg)["결과"])

화면: 주문번호 1001 승인할까요?
실제: 1002번 주문 승인
화면: 주문번호 1003 승인할까요?
실제: 1003번 주문 승인


## [필수 실습] 위키독스 https://wikidocs.net/319029,Time Travel - 과거 상태로 이동

## 번역 파이프라인 예제의 특징

**LLM을 세 번 나눠 쓰는 순차 파이프라인 + 단계별 체크포인트**

- **단계 분리**: 번역 → 검수 → 정리를 각각 노드로 나눴다. 한 노드는 한 가지 일만 한다.
- **State로 결과 전달**: 앞 노드의 결과(`translated`, `review`)를 State에 넣으면 다음 노드가 꺼내 쓴다.
- **덮어쓰기 State**: 리듀서가 없다. 각 키를 한 노드만 채우니 쌓을 필요가 없다.
- **LLM 노드와 일반 노드**: `translate`, `review`는 LLM을 부르고, `finalize`는 문자열만 합친다.
- **체크포인터**: 노드가 하나 끝날 때마다 State가 저장된다.
  - `get_state_history`로 **단계별 중간 결과**를 볼 수 있다.
  - `update_state`로 번역문을 고친 뒤 **검수부터 다시** 실행할 수 있다.

> 핵심: 번역이 마음에 안 들면 **그 단계만 고쳐서 이어서 실행**할 수 있다.


## 이 코드의 장점

1. **중간 결과가 남는다**
   체크포인터 덕분에 번역문, 검수 결과가 단계별로 저장된다. `get_state_history`로 "어느 단계에서 이상해졌는지" 볼 수 있다.

2. **한 단계만 고쳐서 다시 돌릴 수 있다**
   번역문을 `update_state`로 고치면 검수부터 다시 실행한다. 앞 단계의 LLM은 다시 부르지 않으니 시간과 비용이 절약된다.

3. **단계별로 바꾸기 쉽다**
   번역은 비싼 모델, 검수는 싼 모델처럼 노드마다 다른 모델이나 프롬프트를 쓸 수 있다. 하나를 바꿔도 다른 노드는 건드리지 않는다.

4. **확장하기 쉽다**
   "검수 불통과면 다시 번역" 같은 분기, 사람 승인 단계를 노드와 엣지만 추가해서 붙일 수 있다.

> 이 코드는 **지금 더 좋은 결과를 내는 코드가 아니라, 나중에 고치고 늘리기 좋은 구조**를 보여 주는 코드다.